In [1]:
# get imports and dependencies
import pandas as pd
import numpy as np
#load the required data of interest
data = pd.read_csv('/Users/aaronmcdonald/hm-fashion-recommender/data/processed/transactions_train_filtered.csv')

In [2]:
print('Dimensions of the data:', data.shape)

data.head()

Dimensions of the data: (3943069, 32)


,t_dat,customer_id,article_id,price,sales_channel_id,club_member_status,age,postal_code,product_code,prod_name,...,department_name,index_code,index_name,index_group_no,index_group_name,section_no,section_name,garment_group_no,garment_group_name,detail_desc
0,2019-09-28,0000f1c71aafe5963c3d195cf273f7bfd50bbf17761c91...,513512003,0.108458,2,ACTIVE,30.0,7c902dca60ee0bd0f9030eefd445d11146e8d24835738a...,513512,Peter softshell parka,...,Men Sport Tops,S,Sport,26,Sport,22,Men H&M Sport,1007,Outdoor,Parka in a sturdy weave with a drawstring hood...
1,2019-09-28,0000f1c71aafe5963c3d195cf273f7bfd50bbf17761c91...,535035001,0.094898,2,ACTIVE,30.0,7c902dca60ee0bd0f9030eefd445d11146e8d24835738a...,535035,Ultimate parka,...,Outdoor/Blazers DS,D,Divided,2,Divided,58,Divided Selected,1007,Outdoor,Padded parka with a pile-lined hood with a tab...
2,2019-09-28,0000f1c71aafe5963c3d195cf273f7bfd50bbf17761c91...,677930066,0.013542,2,ACTIVE,30.0,7c902dca60ee0bd0f9030eefd445d11146e8d24835738a...,677930,Queen Sweater,...,Basic 1,D,Divided,2,Divided,51,Divided Basics,1002,Jersey Basic,Top in lightweight sweatshirt fabric with ribb...
3,2019-11-08,0000f1c71aafe5963c3d195cf273f7bfd50bbf17761c91...,771528003,0.047712,2,ACTIVE,30.0,7c902dca60ee0bd0f9030eefd445d11146e8d24835738a...,771528,Jeff sweater,...,Knitwear,F,Menswear,3,Menswear,21,Contemporary Casual,1003,Knitwear,Long-sleeved jumper in a soft knit with ribbin...
4,2019-11-08,0000f1c71aafe5963c3d195cf273f7bfd50bbf17761c91...,793856005,0.047712,2,ACTIVE,30.0,7c902dca60ee0bd0f9030eefd445d11146e8d24835738a...,793856,Grape tee,...,Jersey,A,Ladieswear,1,Ladieswear,18,Womens Trend,1005,Jersey Fancy,T-shirt in jersey made from a silk and cotton ...


In [6]:
# see how many unique customers there are 
print('There are', data['customer_id'].nunique(), 'unique customers in the dataset')

# see how many unique articles there are 
print('There are', data['article_id'].nunique(), 'unique articles of clothing in the dataset')

# see the number of transactions per customer
print(data['customer_id'].value_counts())

There are 56595 unique customers in the dataset
There are 60090 unique articles of clothing in the dataset
customer_id
be1981ab818cf4ef6765b2ecaea7a2cbf14ccd6e8a7ee985513d9e8e53c6d91b    1020
b4db5e5259234574edfff958e170fe3a5e13b6f146752ca066abca3c156acc71     803
55d15396193dfd45836af3a6269a079efea339e875eff42cc0c228b002548a9d     690
7f0ac4394297dc4a885d3b9277ba526cbbfbf7fb7cae465b256ed8e55b864f03     686
49beaacac0c7801c2ce2d189efe525fe80b5d37e46ed05b50a4cd88e34d0748f     672
                                                                    ... 
e3ac66fa7b8468b07b10fb8320d1926c6495c500fa4211abeb172599990d3e4f      40
83628ae46e2e11e100b31468b01bce0688f311d24c358609648bdc85dd045db3      40
2a1888700346d71a64a5fb5259b1e2cea43fcf9eda4d158945d1ea842b64780d      40
2a149c0ea0c4e042779471c644ed23f9f647103117488e6fe93def8b30a367e9      40
316a37608a6e9b72501a39371db4af9240e527ac3dec4cf08d5df6efe3e2a853      40
Name: count, Length: 56595, dtype: int64


In [9]:
# split the data into training and testing sets for the LSTM model
# we need the last 7 transactions worth of data for the testing set

# Step 1: group transactions per customer

# ensure the data is sorted by customer_id and t_dat then group by customer_id, getting groups
customer_groups = data.sort_values(by=['customer_id', 't_dat']).groupby('customer_id')

# create a dictionary of customer_id as keys and their corresponding list of bought articles of clothing
customer_sequences = customer_groups['article_id'].apply(list)

print('The first 5 customers and their corresponding list of bought articles of clothing are:')
customer_sequences.head()



The first 5 customers and their corresponding list of bought articles of clothing are:


customer_id
0000f1c71aafe5963c3d195cf273f7bfd50bbf17761c9199e53dbb81641becd7    [513512003, 535035001, 677930066, 771528003, 7...
00023e3dd8618bc63ccad995a5ac62e21177338d642d66b42e0038c6b10f655a    [755609003, 685814001, 685814020, 663009004, 7...
0006d3ff0caf0cb4d4e0615ee5cb7d268622364d483335bf21fbf296e785e282    [775686002, 789707005, 658298001, 658298001, 8...
00075ef36696a7b4ed8c83e22a4bf7ea7c90ee110991ec5e0fb21b12f862f73d    [649397012, 649397012, 790132001, 790132001, 7...
00077dbd5c4a4991e092e63893ccf29294a9d5c46e85010e95f2fc10bf9437a4    [763284006, 826492007, 831686003, 714790020, 8...
Name: article_id, dtype: object

In [15]:
# step 2: train/test split per customer
# ideally we want the last 7 transactions worth of data for the testing set

train_seq = []
test_seq = []

for articles in customer_sequences:
    train_seq.append(articles[:-7])
    test_seq.append(articles[-7:])

X, Y = [], []

for seq in train_seq:
    for i in range(len(seq) - 14):
        X.append(seq[i:i+7])
        Y.append(seq[i+7:i+14])

In [16]:
from sklearn.preprocessing import LabelEncoder

# Flatten all articles into one list
all_articles = list(set([item for sublist in X+Y for item in sublist]))

# Fit encoder
encoder = LabelEncoder()
encoder.fit(all_articles)

# Encode sequences
X_encoded = [encoder.transform(x) for x in X]
Y_encoded = [encoder.transform(y) for y in Y]

# Save reverse mapping for decoding
idx_to_article = dict(zip(encoder.transform(encoder.classes_), encoder.classes_))


In [18]:
import torch

X_tensor = torch.tensor(X_encoded, dtype=torch.long)
Y_tensor = torch.tensor(Y_encoded, dtype=torch.long)


/var/folders/_0/dn5dccc95bqg3t95092yr1rr0000gn/T/ipykernel_35973/3320104552.py:3: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /Users/runner/work/_temp/anaconda/conda-bld/pytorch_1729647038473/work/torch/csrc/utils/tensor_new.cpp:281.)
  X_tensor = torch.tensor(X_encoded, dtype=torch.long)
